In [1]:
pip install boto3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.7/14.7 MB 105.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 9.8 MB/s eta 0:00:00


In [6]:
import os
import hashlib
from typing import List, Dict, Iterable

import boto3
import pyarrow.parquet as pq
from botocore.config import Config
from botocore.exceptions import ClientError
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer
from google.colab import userdata


# ================= CONFIG =================
AWS_ACCESS_KEY_ID = userdata.get("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = userdata.get("AWS_SECRET_ACCESS_KEY")
AWS_REGION = userdata.get("AWS_DEFAULT_REGION") or "us-east-2"

SOURCE_BUCKET = "cleaned-legal-data-analysis-src-v2"
PARQUET_KEY = "pile_of_law/cleaned_contracts_4.parquet"

TARGET_VECTOR_BUCKET = "combined.embeddings"
TARGET_INDEX_NAME = "legal-combined-index"

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

MAX_TOKENS = 350
OVERLAP = 75

EMBED_BATCH = 512
BUFFER_SIZE = 2000
PUT_BATCH = 100
GET_EXISTING_BATCH = 100

LOCAL_PARQUET_PATH = "/tmp/pile_of_law_current.parquet"


# ================= AWS SESSION =================
session = boto3.Session(
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    region_name=AWS_REGION,
)

cfg = Config(retries={"max_attempts": 10, "mode": "standard"})
s3 = session.client("s3", config=cfg)
vec = session.client("s3vectors", config=cfg)


# ================= MODEL =================
print("Loading model on GPU...")
model = SentenceTransformer(MODEL_NAME, device="cuda")
model = model.half()
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


# ================= HELPERS =================
def clean_text(x) -> str:
    return " ".join(str(x).replace("\x00", " ").split()).strip()


def chunk_text(text: str) -> List[str]:
    text = clean_text(text)
    if not text:
        return []

    token_ids = tokenizer(
        text,
        add_special_tokens=False,
        truncation=False,
        return_attention_mask=False,
        return_token_type_ids=False,
    )["input_ids"]

    if not token_ids:
        return []

    if len(token_ids) <= MAX_TOKENS:
        chunk = tokenizer.decode(token_ids, skip_special_tokens=True).strip()
        return [chunk] if chunk else []

    chunks = []
    step = MAX_TOKENS - OVERLAP

    for start in range(0, len(token_ids), step):
        end = start + MAX_TOKENS
        piece = token_ids[start:end]
        if not piece:
            continue

        chunk = tokenizer.decode(piece, skip_special_tokens=True).strip()
        if chunk:
            chunks.append(chunk)

        if end >= len(token_ids):
            break

    return chunks


def make_vector_key(
    dataset: str,
    parquet_key: str,
    row_group_idx: int,
    row_idx: int,
    chunk_idx: int,
    doc_id: str,
    chunk_text_value: str,
) -> str:
    base = f"{dataset}|{parquet_key}|rg={row_group_idx}|row={row_idx}|chunk={chunk_idx}|doc={doc_id}"
    base_hash = hashlib.md5(base.encode("utf-8")).hexdigest()[:20]
    text_hash = hashlib.md5(chunk_text_value.encode("utf-8")).hexdigest()[:12]
    return f"{dataset}::{base_hash}::{text_hash}"


def batched(items: List, size: int) -> Iterable[List]:
    for i in range(0, len(items), size):
        yield items[i:i + size]


def ensure_vector_bucket_and_index() -> None:
    bucket_exists = True
    try:
        vec.get_vector_bucket(vectorBucketName=TARGET_VECTOR_BUCKET)
    except ClientError as e:
        code = e.response.get("Error", {}).get("Code", "")
        if code in {"NotFoundException", "ResourceNotFoundException"}:
            bucket_exists = False
        else:
            raise

    if not bucket_exists:
        vec.create_vector_bucket(vectorBucketName=TARGET_VECTOR_BUCKET)

    index_exists = True
    try:
        vec.get_index(
            vectorBucketName=TARGET_VECTOR_BUCKET,
            indexName=TARGET_INDEX_NAME,
        )
    except ClientError as e:
        code = e.response.get("Error", {}).get("Code", "")
        if code in {"NotFoundException", "ResourceNotFoundException"}:
            index_exists = False
        else:
            raise

    if not index_exists:
        vec.create_index(
            vectorBucketName=TARGET_VECTOR_BUCKET,
            indexName=TARGET_INDEX_NAME,
            dataType="float32",
            dimension=384,
            distanceMetric="cosine",
        )


def existing_keys(keys: List[str]) -> set:
    found = set()

    for batch in batched(keys, GET_EXISTING_BATCH):
        try:
            resp = vec.get_vectors(
                vectorBucketName=TARGET_VECTOR_BUCKET,
                indexName=TARGET_INDEX_NAME,
                keys=batch,
            )
            for item in resp.get("vectors", []):
                k = item.get("key")
                if k:
                    found.add(k)
        except ClientError:
            pass

    return found


def put_vectors(vectors: List[Dict]) -> None:
    for batch in batched(vectors, PUT_BATCH):
        vec.put_vectors(
            vectorBucketName=TARGET_VECTOR_BUCKET,
            indexName=TARGET_INDEX_NAME,
            vectors=batch,
        )


def flush_buffer(buffer: List[Dict], stats: Dict[str, int]) -> List[Dict]:
    if not buffer:
        return []

    unique_map = {}
    for row in buffer:
        unique_map[row["key"]] = row
    unique_rows = list(unique_map.values())

    stats["attempted"] += len(unique_rows)

    keys = [row["key"] for row in unique_rows]
    already = existing_keys(keys)

    to_insert = [row for row in unique_rows if row["key"] not in already]
    skipped = len(unique_rows) - len(to_insert)
    stats["skipped"] += skipped

    if not to_insert:
        print(
            f"Inserted={stats['inserted']}  "
            f"Skipped={stats['skipped']}  "
            f"Attempted={stats['attempted']}"
        )
        return []

    texts = [row["text"] for row in to_insert]
    embeddings = model.encode(
        texts,
        batch_size=EMBED_BATCH,
        normalize_embeddings=True,
        show_progress_bar=False,
    )

    payload = []
    seen_payload = set()

    for row, emb in zip(to_insert, embeddings):
        key = row["key"]
        if key in seen_payload:
            continue
        seen_payload.add(key)

        payload.append(
            {
                "key": key,
                "data": {"float32": [float(x) for x in emb]},
                "metadata": {
                    "dataset": "pile_of_law",
                    "doc_id": row["doc_id"][:200],
                    "source_uri": row["source_uri"][:500],
                },
            }
        )

    if payload:
        put_vectors(payload)
        stats["inserted"] += len(payload)

    print(
        f"Inserted={stats['inserted']}  "
        f"Skipped={stats['skipped']}  "
        f"Attempted={stats['attempted']}"
    )
    return []


# ================= MAIN =================
def main():
    if not AWS_ACCESS_KEY_ID or not AWS_SECRET_ACCESS_KEY:
        raise ValueError("AWS credentials not found in Colab secrets.")

    print("Ensuring target vector bucket and index exist...")
    ensure_vector_bucket_and_index()

    print(f"Downloading parquet from s3://{SOURCE_BUCKET}/{PARQUET_KEY}")
    s3.download_file(SOURCE_BUCKET, PARQUET_KEY, LOCAL_PARQUET_PATH)
    print(f"Local file ready: {LOCAL_PARQUET_PATH}")

    parquet_file = pq.ParquetFile(LOCAL_PARQUET_PATH)

    stats = {
        "row_groups": parquet_file.num_row_groups,
        "rows_seen": 0,
        "chunks_seen": 0,
        "attempted": 0,
        "inserted": 0,
        "skipped": 0,
    }

    buffer = []
    source_uri = f"s3://{SOURCE_BUCKET}/{PARQUET_KEY}"

    print("Processing parquet row groups...")
    for rg_idx in range(parquet_file.num_row_groups):
        table = parquet_file.read_row_group(rg_idx)
        rows = table.to_pylist()

        for row_idx, row in enumerate(rows):
            stats["rows_seen"] += 1

            text = clean_text(
                row.get("text")
                or row.get("cleaned_text")
                or row.get("content")
                or row.get("body")
                or ""
            )
            if not text:
                continue

            doc_id = clean_text(
                row.get("doc_id")
                or row.get("document_id")
                or row.get("contract_id")
                or row.get("file_id")
                or row.get("id")
                or f"doc_{rg_idx}_{row_idx}"
            )

            chunks = chunk_text(text)
            if not chunks:
                continue

            for chunk_idx, chunk_value in enumerate(chunks):
                stats["chunks_seen"] += 1
                key = make_vector_key(
                    dataset="pile_of_law",
                    parquet_key=PARQUET_KEY,
                    row_group_idx=rg_idx,
                    row_idx=row_idx,
                    chunk_idx=chunk_idx,
                    doc_id=doc_id,
                    chunk_text_value=chunk_value,
                )

                buffer.append(
                    {
                        "key": key,
                        "doc_id": doc_id,
                        "text": chunk_value,
                        "source_uri": source_uri,
                    }
                )

            if len(buffer) >= BUFFER_SIZE:
                buffer = flush_buffer(buffer, stats)

    if buffer:
        buffer = flush_buffer(buffer, stats)

    print("\nDone.")
    print(f"Row groups       : {stats['row_groups']}")
    print(f"Rows seen        : {stats['rows_seen']}")
    print(f"Chunks seen      : {stats['chunks_seen']}")
    print(f"Attempted chunks : {stats['attempted']}")
    print(f"Inserted chunks  : {stats['inserted']}")
    print(f"Skipped existing : {stats['skipped']}")


if __name__ == "__main__":
    main()

Loading model on GPU...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Ensuring target vector bucket and index exist...
Local file ready: /tmp/pile_of_law_current.parquet
Processing parquet row groups...


Token indices sequence length is longer than the specified maximum sequence length for this model (1204 > 512). Running this sequence through the model will result in indexing errors


Inserted=2001  Skipped=0  Attempted=2001
Inserted=4004  Skipped=0  Attempted=4004
Inserted=6006  Skipped=0  Attempted=6006
Inserted=8010  Skipped=0  Attempted=8010
Inserted=10010  Skipped=0  Attempted=10010
Inserted=12013  Skipped=0  Attempted=12013
Inserted=14016  Skipped=0  Attempted=14016
Inserted=16016  Skipped=0  Attempted=16016
Inserted=18017  Skipped=0  Attempted=18017
Inserted=20019  Skipped=0  Attempted=20019
Inserted=22019  Skipped=0  Attempted=22019
Inserted=24019  Skipped=0  Attempted=24019
Inserted=26019  Skipped=0  Attempted=26019
Inserted=28022  Skipped=0  Attempted=28022
Inserted=30024  Skipped=0  Attempted=30024
Inserted=32027  Skipped=0  Attempted=32027
Inserted=34028  Skipped=0  Attempted=34028
Inserted=36032  Skipped=0  Attempted=36032
Inserted=38034  Skipped=0  Attempted=38034
Inserted=40035  Skipped=0  Attempted=40035
Inserted=42038  Skipped=0  Attempted=42038
Inserted=44038  Skipped=0  Attempted=44038
Inserted=46041  Skipped=0  Attempted=46041
Inserted=48045  Ski